In [0]:
from pyspark.sql.functions import col, to_date

fact_df = spark.table("medical_project.gold.fact_encounters")
date_df = spark.table("medical_project.gold.dim_date")

display(fact_df)


In [0]:
# Join with Date Dimension
fact_with_date = fact_df.withColumn(
    "encounter_date",
    to_date(col("start"))
).join(
    date_df,
    col("encounter_date") == col("date"),
    "left"
)

In [0]:
# Create Duration Category
from pyspark.sql.functions import when

fact_with_date = fact_with_date.withColumn(
    "duration_category",
    when(col("encounter_duration_hours") > 24, "More than 24 hours")
    .otherwise("24 hours or less")
)

display(fact_with_date)

In [0]:
# Aggregate Counts
from pyspark.sql.functions import count

agg_df = fact_with_date.groupBy(
    "year", "month", "duration_category"
).agg(
    count("encounter_id").alias("encounter_count")
)

In [0]:
# Total Encounters per Month
from pyspark.sql.window import Window
from pyspark.sql.functions import sum

window_spec = Window.partitionBy("year", "month")

agg_df = agg_df.withColumn(
    "total_encounters",
    sum("encounter_count").over(window_spec)
)

# Calculate Percentage
agg_df = agg_df.withColumn(
    "percentage",
    (col("encounter_count") / col("total_encounters")) * 100
)

display(agg_df)

In [0]:
# Final Output
kpi2 = agg_df.select(
    "year",
    "month",
    "duration_category",
    "encounter_count",
    "percentage"
).orderBy("year", "month")

display(kpi2)

In [0]:
# Save Table
kpi2.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.gold.kpi_encounter_duration_split")